# LogiSense — 03 | OLAP Cube, Slice/Dice, Roll-up/Drill-down & Pivot

### Objective
Create the analytical engine that powers the website's **OLAP Explorer**.

### DWM topics
**OLAP · cube thinking · dimensions · measures · Slice · Dice · Roll-up · Drill-down · Pivot · aggregation · dynamic filters · SQL analytical queries**

In [1]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA=Path("../data/processed")
if (DATA/"fact_delivery.csv").exists():
    fact=pd.read_csv(DATA/"fact_delivery.csv")
    tables={n:pd.read_csv(DATA/f"{n}.csv") for n in [
        "dim_date","dim_customer","dim_product","dim_warehouse","dim_carrier","dim_shipping","dim_weather"
    ] if (DATA/f"{n}.csv").exists()}
    cube=fact.copy()
    for n,t in tables.items():
        key = [c for c in t.columns if c.endswith("_key")][0]
        if key in cube.columns:
            cube=cube.merge(t,on=key,how="left")
else:
    csvs=list(Path("../data/raw").glob("*.csv"))
    if len(csvs)!=1: raise FileNotFoundError("Run Notebook 02 first or place the raw CSV in data/raw.")
    cube=pd.read_csv(csvs[0])

print("Analytical cube rows:",len(cube))
display(cube.head(2))

Analytical cube rows: 50000


,delivery_key,order_id,date_key,customer_key,product_key,warehouse_key,carrier_key,shipping_key,weather_key,product_weight_kg,...,customer_country,product_category,package_size,warehouse_id,warehouse_city,carrier,shipping_method,payment_method,order_priority,weather_condition
0,1,ORD-000001,319,1,1,1,1,1,1,8.23,...,Japan,Home & Kitchen,Large,WH-004,Toronto,EagleCourier,International,Digital Wallet,Normal,Snow
1,2,ORD-000002,98,2,2,2,2,2,2,0.10,...,Canada,Beauty,Small,WH-007,Paris,BlueRoute,Standard,Debit Card,Normal,Cloudy


## 1. Define the semantic layer

A real analytics application should not let the browser send arbitrary SQL.  
Instead, it exposes a safe list of dimensions and measures and translates selections into controlled group-bys.

In [2]:
DIMENSIONS={
    "Country":"customer_country","Customer Segment":"customer_segment",
    "Warehouse":"warehouse_city","Carrier":"carrier",
    "Product Category":"product_category","Shipping Method":"shipping_method",
    "Weather":"weather_condition","Priority":"order_priority",
    "Year":"year","Quarter":"quarter","Month":"month_name"
}
MEASURES={
    "Orders":("order_id","count"),
    "Order Value":("order_value_usd","sum"),
    "Shipping Cost":("shipping_cost_usd","sum"),
    "Avg Delivery Days":("actual_delivery_days","mean"),
    "Avg Delay Days":("delivery_delay_days","mean"),
    "Avg Rating":("customer_rating","mean"),
    "Late Rate":("late_flag","mean"),
    "Return Rate":("return_flag","mean")
}
available_dims={k:v for k,v in DIMENSIONS.items() if v in cube.columns}
available_measures={k:v for k,v in MEASURES.items() if v[0] in cube.columns}
print("Dimensions:",available_dims)
print("Measures:",available_measures)

Dimensions: {'Country': 'customer_country', 'Customer Segment': 'customer_segment', 'Warehouse': 'warehouse_city', 'Carrier': 'carrier', 'Product Category': 'product_category', 'Shipping Method': 'shipping_method', 'Weather': 'weather_condition', 'Priority': 'order_priority', 'Year': 'year', 'Quarter': 'quarter', 'Month': 'month_name'}
Measures: {'Orders': ('order_id', 'count'), 'Order Value': ('order_value_usd', 'sum'), 'Shipping Cost': ('shipping_cost_usd', 'sum'), 'Avg Delivery Days': ('actual_delivery_days', 'mean'), 'Avg Delay Days': ('delivery_delay_days', 'mean'), 'Avg Rating': ('customer_rating', 'mean'), 'Late Rate': ('late_flag', 'mean'), 'Return Rate': ('return_flag', 'mean')}


## 2. OLAP function used by the website

The same function supports the dashboard, filter panel and OLAP Explorer.

In [3]:
def olap_query(data, row_dimension, measure, column_dimension=None, filters=None):
    d=data.copy()
    row_col=available_dims[row_dimension]
    value_col, agg=available_measures[measure]

    for label, selected in (filters or {}).items():
        if label not in available_dims: continue
        col=available_dims[label]
        selected=selected if isinstance(selected,list) else [selected]
        d=d[d[col].isin(selected)]

    if column_dimension:
        col_col=available_dims[column_dimension]
        result=pd.pivot_table(d,index=row_col,columns=col_col,values=value_col,aggfunc=agg)
    else:
        result=d.groupby(row_col)[value_col].agg(agg).to_frame(measure)

    if measure in ["Late Rate","Return Rate"]:
        result=result*100
    return result.sort_values(result.columns[0],ascending=False)

display(olap_query(
    cube,
    "Carrier",
    "Avg Delay Days",
    "Month",
    {"Country":["India"]} if "customer_country" in cube.columns else {}
).round(2))

month_name,April,August,December,February,January,July,June,March,May,November,October,September
carrier,,,,,,,,,,,,
GlobalExpress,2.31,2.14,1.53,2.07,2.39,2.00,2.09,2.77,1.73,2.45,2.06,2.64
PrimeDelivery,2.28,2.20,1.98,1.61,2.33,1.89,1.59,2.26,2.02,2.31,2.13,2.92
EagleCourier,1.94,1.29,1.56,1.45,1.48,1.82,2.00,1.28,1.92,1.29,1.94,2.00
BlueRoute,1.69,2.32,1.88,2.12,2.21,2.02,1.56,2.18,2.17,2.30,1.69,2.06
FastTrack Logistics,1.55,1.66,1.77,1.45,1.39,1.89,2.31,1.80,1.57,1.67,1.46,1.62
ParcelPro,1.50,1.48,1.67,1.59,1.31,1.73,1.48,1.64,1.87,1.76,1.59,1.84
SpeedyCargo,1.48,1.23,1.80,2.17,2.11,1.64,1.94,1.72,1.82,1.61,1.74,1.93
SwiftShip,1.14,1.90,1.61,1.21,1.39,1.62,1.42,1.33,1.44,1.41,1.35,1.54


## 3. SLICE

**Slice = fix one dimension to a single value.**

Business question: *How is India performing compared with the overall business?*

In [4]:
if "customer_country" in cube.columns:
    overall = cube["delivery_delay_days"].mean()
    india = cube.loc[cube.customer_country.eq("India"),"delivery_delay_days"].mean()
    print(f"Overall average delay: {overall:.2f}")
    print(f"India average delay:   {india:.2f}")
    print(f"Difference:            {india-overall:+.2f} days")

Overall average delay: 1.28
India average delay:   1.81
Difference:            +0.53 days


## 4. DICE

**Dice = filter multiple dimensions simultaneously.**

Example: India + selected product categories + selected shipping methods.

In [5]:
filters={}
if "customer_country" in cube.columns: filters["Country"]=["India"]
if "product_category" in cube.columns:
    filters["Product Category"]=cube.product_category.dropna().unique()[:3].tolist()
if "shipping_method" in cube.columns:
    filters["Shipping Method"]=cube.shipping_method.dropna().unique()[:2].tolist()

display(olap_query(cube,"Carrier","Late Rate",filters=filters).round(2))

,Late Rate
carrier,
GlobalExpress,81.55
PrimeDelivery,77.89
BlueRoute,72.64
SpeedyCargo,70.97
ParcelPro,67.33
FastTrack Logistics,66.67
EagleCourier,66.00
SwiftShip,61.80


## 5. ROLL-UP

**Roll-up = summarize to a higher level in a hierarchy.**

Time hierarchy:
`Day → Month → Quarter → Year`

In [6]:
if all(c in cube.columns for c in ["year","quarter","month"]):
    rollup = cube.groupby(["year","quarter","month"],as_index=False).agg(
        orders=("order_id","count"),
        revenue=("order_value_usd","sum"),
        avg_delay=("delivery_delay_days","mean"),
        shipping_cost=("shipping_cost_usd","sum"),
        late_rate=("late_flag","mean")
    )
    rollup["late_rate"]*=100
    display(rollup.round(2))

,year,quarter,month,orders,revenue,avg_delay,shipping_cost,late_rate
0,2026,Q1,1,4303,761504.76,1.26,512637.66,56.73
1,2026,Q1,2,3859,697000.25,1.27,460213.46,56.18
2,2026,Q1,3,4219,748637.10,1.36,527435.61,57.74
3,2026,Q2,4,4090,723645.03,1.24,483845.30,55.48
4,2026,Q2,5,4253,747052.64,1.24,506816.38,55.49
5,2026,Q2,6,4118,725204.30,1.27,492372.54,56.14
6,2026,Q3,7,4319,763134.09,1.30,527849.83,56.73
7,2026,Q3,8,4284,764872.41,1.27,509970.22,55.70
8,2026,Q3,9,4030,704777.36,1.32,492678.51,56.13
9,2026,Q4,10,4170,741826.21,1.28,498348.03,56.59


## 6. DRILL-DOWN

**Drill-down = move from summary to greater detail.**

The website can implement this as:
Year → click a year → Quarter → click a quarter → Month → Day.

In [7]:
levels=[c for c in ["year","quarter","month","month_name","order_date"] if c in cube.columns]
if levels:
    drill=cube.groupby(levels,as_index=False).agg(
        orders=("order_id","count"),
        avg_delay=("delivery_delay_days","mean"),
        late_rate=("late_flag","mean")
    )
    drill["late_rate"]*=100
    display(drill.head(25).round(2))

,year,quarter,month,month_name,order_date,orders,avg_delay,late_rate
0,2026,Q1,1,January,2026-01-01,141,1.33,56.74
1,2026,Q1,1,January,2026-01-02,130,0.97,48.46
2,2026,Q1,1,January,2026-01-03,131,1.43,61.83
3,2026,Q1,1,January,2026-01-04,140,1.02,46.43
4,2026,Q1,1,January,2026-01-05,144,1.28,58.33
5,2026,Q1,1,January,2026-01-06,155,1.23,58.71
6,2026,Q1,1,January,2026-01-07,137,1.45,57.66
7,2026,Q1,1,January,2026-01-08,140,1.28,56.43
8,2026,Q1,1,January,2026-01-09,131,1.32,54.96
9,2026,Q1,1,January,2026-01-10,148,1.30,61.49


## 7. PIVOT

**Pivot = rotate a dimension from rows into columns.**

This is useful for comparing carriers across months or warehouses across shipping methods.

In [8]:
if "carrier" in cube.columns and "month_name" in cube.columns:
    pivot=pd.pivot_table(cube,index="carrier",columns="month_name",
                         values="delivery_delay_days",aggfunc="mean")
    display(pivot.round(2))

month_name,April,August,December,February,January,July,June,March,May,November,October,September
carrier,,,,,,,,,,,,
BlueRoute,1.28,1.50,1.54,1.47,1.47,1.43,1.38,1.58,1.44,1.38,1.34,1.39
EagleCourier,1.14,1.21,1.29,1.22,1.14,1.28,1.24,1.26,1.19,1.10,1.16,1.28
FastTrack Logistics,1.15,1.18,1.25,1.21,1.29,1.30,1.24,1.50,1.17,1.25,1.29,1.26
GlobalExpress,1.74,1.61,1.59,1.61,1.55,1.65,1.65,1.75,1.43,1.63,1.56,1.64
ParcelPro,1.07,1.08,1.06,0.98,1.15,1.22,1.21,1.19,1.18,1.15,1.25,1.22
PrimeDelivery,1.40,1.46,1.55,1.40,1.47,1.46,1.43,1.59,1.38,1.46,1.47,1.61
SpeedyCargo,1.17,1.11,1.20,1.15,1.07,1.09,1.04,1.09,1.12,1.15,1.16,1.15
SwiftShip,0.93,1.00,0.92,1.04,0.98,0.98,0.99,0.94,0.97,0.99,1.01,1.09


## 8. Comparison #1 — Carrier scorecard

A useful company decision is not simply "which carrier has the lowest average delay?"  
We compare **reliability + cost + customer experience + volume**.

In [9]:
if "carrier" in cube.columns:
    carrier_score=cube.groupby("carrier").agg(
        orders=("order_id","count"),
        avg_delay=("delivery_delay_days","mean"),
        late_rate=("late_flag","mean"),
        avg_shipping_cost=("shipping_cost_usd","mean"),
        avg_rating=("customer_rating","mean")
    )
    carrier_score["late_rate"]*=100
    # Normalize directionally for an illustrative composite score
    for c in ["avg_delay","late_rate","avg_shipping_cost"]:
        lo,hi=carrier_score[c].min(),carrier_score[c].max()
        carrier_score[c+"_norm"]=(carrier_score[c]-lo)/(hi-lo+1e-9)
    carrier_score["ops_score"] = (
        0.40*(1-carrier_score["late_rate_norm"]) +
        0.30*(1-carrier_score["avg_delay_norm"]) +
        0.20*(1-carrier_score["avg_shipping_cost_norm"]) +
        0.10*(carrier_score["avg_rating"]/5)
    )*100
    display(carrier_score.sort_values("ops_score",ascending=False).round(3))

,orders,avg_delay,late_rate,avg_shipping_cost,avg_rating,avg_delay_norm,late_rate_norm,avg_shipping_cost_norm,ops_score
carrier,,,,,,,,,
SwiftShip,6233,0.987,47.345,120.186,3.503,0.000,0.000,0.358,89.838
ParcelPro,6374,1.149,52.055,118.843,3.435,0.257,0.256,0.040,78.107
EagleCourier,6213,1.210,54.869,119.604,3.393,0.355,0.409,0.220,65.390
SpeedyCargo,6257,1.123,51.526,122.894,3.450,0.217,0.227,1.000,61.290
FastTrack Logistics,6072,1.258,56.571,121.627,3.371,0.431,0.501,0.700,49.771
BlueRoute,6435,1.434,61.026,121.383,3.302,0.711,0.743,0.642,32.706
PrimeDelivery,6185,1.472,61.455,120.943,3.290,0.771,0.767,0.538,32.019
GlobalExpress,6231,1.616,65.752,118.674,3.231,1.000,1.000,0.000,26.462


## 9. Comparison #2 — Warehouse benchmark

Compare warehouses by volume, delay, processing time and late rate.

**Interpretation rule:** a warehouse with low delay but very low volume is not automatically the best operation. Always consider scale.

In [10]:
if "warehouse_city" in cube.columns:
    wh=cube.groupby("warehouse_city").agg(
        orders=("order_id","count"),
        avg_processing_hours=("warehouse_processing_hours","mean"),
        avg_delay=("delivery_delay_days","mean"),
        late_rate=("late_flag","mean"),
        avg_shipping_cost=("shipping_cost_usd","mean")
    )
    wh["late_rate"]*=100
    display(wh.sort_values("orders",ascending=False).round(2))

,orders,avg_processing_hours,avg_delay,late_rate,avg_shipping_cost
warehouse_city,,,,,
New York,7597,14.51,1.31,56.75,117.76
Los Angeles,7534,14.42,1.25,55.75,120.79
Chicago,5960,14.31,1.28,56.02,119.32
Toronto,4977,14.60,1.27,56.10,120.84
Berlin,4931,14.46,1.28,56.15,122.22
London,4923,14.40,1.27,57.00,120.38
Paris,3990,14.50,1.31,57.12,122.02
Sydney,3980,14.13,1.32,57.59,123.39
Dubai,3606,14.41,1.26,55.88,124.06


## 10. Comparison #3 — Shipping method trade-off

Compare speed, reliability and cost. This is a **business trade-off**, not a single-metric ranking.

In [11]:
if "shipping_method" in cube.columns:
    sm=cube.groupby("shipping_method").agg(
        orders=("order_id","count"),
        avg_delivery=("actual_delivery_days","mean"),
        avg_delay=("delivery_delay_days","mean"),
        late_rate=("late_flag","mean"),
        avg_cost=("shipping_cost_usd","mean"),
        avg_rating=("customer_rating","mean")
    )
    sm["late_rate"]*=100
    display(sm.round(2))

,orders,avg_delivery,avg_delay,late_rate,avg_cost,avg_rating
shipping_method,,,,,,
Economy,5577,10.56,1.50,61.70,48.91,3.28
Express,9696,3.61,0.62,43.41,107.58,3.57
International,15839,13.51,2.08,67.67,204.40,3.20
Same Day,1861,1.42,0.29,24.50,136.58,3.84
Standard,17027,6.28,0.96,54.84,71.55,3.40


## 11. SQL examples for the backend

The Flask/FastAPI backend should generate parameterized queries from the same semantic layer.

### Carrier roll-up
```sql
SELECT c.carrier,
       COUNT(*) AS orders,
       AVG(f.delivery_delay_days) AS avg_delay,
       AVG(f.late_flag) * 100 AS late_rate
FROM fact_delivery f
JOIN dim_carrier c ON f.carrier_key = c.carrier_key
GROUP BY c.carrier
ORDER BY late_rate DESC;
```

### Slice + Dice
```sql
SELECT p.product_category,
       s.shipping_method,
       COUNT(*) AS orders,
       AVG(f.delivery_delay_days) AS avg_delay
FROM fact_delivery f
JOIN dim_customer c ON f.customer_key=c.customer_key
JOIN dim_product p ON f.product_key=p.product_key
JOIN dim_shipping s ON f.shipping_key=s.shipping_key
WHERE c.customer_country = ?
  AND p.product_category IN (?, ?)
  AND s.shipping_method IN (?, ?)
GROUP BY p.product_category, s.shipping_method;
```

The `?` placeholders are intentional: use parameterized SQL, not string concatenation.

## 12. Website mapping

| OLAP concept | Website feature |
|---|---|
| Slice | Country / carrier / warehouse filter |
| Dice | Multiple filters together |
| Roll-up | Year / quarter / month summary |
| Drill-down | Click a chart point to reveal detail |
| Pivot | Rows vs columns selector |
| Measures | KPI dropdown |
| Dimensions | Group-by dropdown |
| Saved query | Save user's filter combination |

This is the core differentiator of LogiSense: the website is an **interactive analytical application**, not a static dashboard.